# Day05 下午个人项目：电商用户多维分析

**姓名：** 和桢楠
**专题方向：** A


本Notebook由每名学生独立完成，并随个人项目仓库提交到GitHub。

> 请只修改标有 `TODO` 的区域，不要删除任务说明、检查点、结论区和提交检查。

## 一、实验目标与提交要求

你需要独立完成：

1. 读取并验收第4天清洗后的数据；
2. 计算公共基础指标；
3. 选择一个专题完成单维分析；
4. 完成至少一个双维度交叉分析；
5. 输出三个标准CSV报表；
6. 撰写至少3条结论、1条限制和1项建议；
7. 将Notebook和输出文件提交到个人GitHub仓库。

### 必须遵守的分析边界

- 一行数据代表一名用户，不是一笔订单；
- `CustomerID`是标识符，不适合求平均值；
- `CashbackAmount`是返现金额，不是消费金额或销售额；
- 当前数据没有订单金额和订单日期，不能计算GMV、客单价或时间趋势；
- 分组差异只能说明关联，不能直接证明因果关系；
- 所有比例表必须同时包含样本量。

## 二、专题方向

| 专题 | 推荐字段 | 参考业务问题 |
|---|---|---|
| A 用户生命周期 | `TenureGroup` | 不同生命周期用户的流失和订单行为有何差异？ |
| B 投诉与服务体验 | `Complain`、`SatisfactionScore` | 投诉、满意度与流失存在怎样的关联？ |
| C 品类与订单行为 | `PreferedOrderCat` | 不同偏好品类用户的规模和订单行为有何差异？ |
| D 支付与优惠行为 | `PreferredPaymentMode` | 支付偏好与优惠行为是否存在分组差异？ |
| E 城市与设备行为 | `CityTier`、`PreferredLoginDevice` | 城市、设备与用户活跃或流失有何关联？ |

请选择一个专题作为单维分析主线。双维分析可以在此基础上增加另一个业务维度。

## 任务0：个人配置与运行环境

In [79]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


# =========================
# TODO：和桢楠 A填写个人信息与专题
# =========================
STUDENT_NAME = "和桢楠"
TOPIC = "A"


pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def find_workspace_root(start=None):
    """从当前目录向上寻找项目根目录。"""
    start = Path.cwd() if start is None else Path(start)

    for candidate in [start, *start.parents]:
        data_path = (
            candidate
            / "output"
            / "day04_project"
            / "ecommerce_customer_cleaned.csv"
        )

        if data_path.exists():
            return candidate

    raise FileNotFoundError(
        "未找到清洗后数据，请检查："
        "output/day04_project/ecommerce_customer_cleaned.csv"
    )


ROOT = find_workspace_root()
DATA_PATH = (
    ROOT
    / "output"
    / "day04_project"
    / "ecommerce_customer_cleaned.csv"
)
OUTPUT_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)
print("输入数据：", DATA_PATH)
print("输出目录：", OUTPUT_DIR)

姓名： 和桢楠
专题： A
输入数据： C:\Users\和桢楠\PyCharmMiscProject\output\day04_project\ecommerce_customer_cleaned.csv
输出目录： C:\Users\和桢楠\PyCharmMiscProject\output\day05_analysis


In [80]:
# 检查点0：个人信息与专题配置

assert STUDENT_NAME != "请填写姓名", "请填写STUDENT_NAME"
assert STUDENT_NAME.strip(), "姓名不能为空"

TOPIC = TOPIC.strip().upper()
assert TOPIC in {"A", "B", "C", "D", "E"}, \
    "TOPIC只能填写A、B、C、D或E"

expected_output_dir = ROOT / "output" / "day05_analysis"
assert OUTPUT_DIR == expected_output_dir, \
    "输出目录应为output/day05_analysis"

print("检查点0通过")
print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)

检查点0通过
姓名： 和桢楠
专题： A


### 检查点0完成标志

- [ ] 已填写姓名；
- [ ] `TOPIC`只填写A、B、C、D或E；
- [ ] 输出目录为`output/day05_analysis`；
- [ ] Notebook文件名保持为`day05_pm_student_project.ipynb`。

## 任务1：读取并验收数据（必做）

In [81]:
# 读取第4天清洗后的数据
df = pd.read_csv(DATA_PATH)

print("数据形状：", df.shape)
display(df.head())
print("\n字段类型：")
display(df.dtypes.to_frame("数据类型"))

数据形状： (5630, 22)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,TenureGroup,IsMobileLogin
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93,0-6个月,1
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90,7-12个月,1
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28,7-12个月,1
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07,新用户,1
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60,新用户,1



字段类型：


,数据类型
CustomerID,int64
Churn,int64
Tenure,float64
PreferredLoginDevice,str
CityTier,int64
WarehouseToHome,float64
PreferredPaymentMode,str
Gender,str
HourSpendOnApp,float64
NumberOfDeviceRegistered,int64


In [82]:
# TODO 1：定义需要验收的核心字段
core_cols = [
    "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder",
    # 示例："CustomerID",
]


# TODO 2：完成数据验收表
row_cnt, col_cnt = df.shape
customerid_dup = df["CustomerID"].duplicated().sum()
core_missing = df[core_cols].isna().sum().sum()
churn_unique = sorted(df["Churn"].unique().tolist())
# 至少包含：行数、列数、CustomerID重复数、核心字段缺失数、Churn取值
validation = pd.DataFrame({
    "验收指标": ["总行数", "总列数", "CustomerID重复条数", "核心字段总缺失数", "Churn所有取值"],
    "结果": [row_cnt, col_cnt, customerid_dup, core_missing, churn_unique]
})


# TODO 3：展示验收结果
display(validation)


,验收指标,结果
0,总行数,5630
1,总列数,22
2,CustomerID重复条数,0
3,核心字段总缺失数,0
4,Churn所有取值,"[0, 1]"


In [83]:
# 检查点1：数据结构与核心质量

assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert df.shape == (5630, 22), "数据形状应为(5630, 22)"
assert df["CustomerID"].is_unique, "CustomerID应保持唯一"
assert set(df["Churn"].unique()) == {0, 1}, \
    "Churn应只包含0和1"

required_core_cols = [
    "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder",
]

assert set(required_core_cols).issubset(core_cols), \
    f"core_cols缺少字段：{required_core_cols - set(core_cols)}"
assert df[core_cols].notna().all().all(), \
    "核心分析字段仍存在缺失值"
assert validation is not None, "请完成validation验收表"

print("检查点1通过")

检查点1通过


### 数据粒度说明

请用一句话说明一行数据代表什么：

> TODO：请填写。一位客户的全量消费、活动及流失状态的个人画像记录。


请说明为什么`CustomerID`不能作为普通连续数值求平均：

> TODO：请填写。CustomerID属于身份标识型分类编号，仅用于唯一区分客户，数值大小无数学意义，平均数不具备业务解释价值。

## 任务2：公共基础指标（必做）

请构建`overall_metrics`，至少包含以下10项指标：

1. 用户数；
2. 流失人数；
3. 总体流失率；
4. 平均订单数；
5. 订单数中位数；
6. 平均优惠券使用次数；
7. 平均返现；
8. 平均App使用时长；
9. 平均满意度；
10. 平均距上次下单天数。

输出建议使用“指标、数值”两列的DataFrame。

In [84]:
# TODO：计算公共基础指标
import pandas as pd

# 计算各项指标
user_count = len(df)
churn_count = df[df["Churn"] == 1].shape[0]
churn_rate = churn_count / user_count

avg_order = df["OrderCount"].mean()
med_order = df["OrderCount"].median()

avg_coupon = df["CouponUsed"].mean()
avg_cashback = df["CashbackAmount"].mean()
avg_app_time = df["HourSpendOnApp"].mean()
avg_satisfaction = df["SatisfactionScore"].mean()
avg_last_order_gap = df["DaySinceLastOrder"].mean()

# 构造指标DataFrame
overall_metrics = pd.DataFrame({
    "指标": [
        "用户数",
        "流失人数",
        "总体流失率",
        "平均订单数",
        "订单数中位数",
        "平均优惠券使用次数",
        "平均返现",
        "平均App使用时长",
        "平均满意度",
        "平均距上次下单天数"
    ],
    "数值": [
        user_count,
        churn_count,
        churn_rate,
        avg_order,
        med_order,
        avg_coupon,
        avg_cashback,
        avg_app_time,
        avg_satisfaction,
        avg_last_order_gap
    ]
})


# TODO：展示结果
display(overall_metrics)

,指标,数值
0,用户数,"5,630.00"
1,流失人数,948.00
2,总体流失率,0.17
3,平均订单数,2.96
4,订单数中位数,2.00
5,平均优惠券使用次数,1.72
6,平均返现,177.22
7,平均App使用时长,2.93
8,平均满意度,3.07
9,平均距上次下单天数,4.46


In [85]:
# 检查点2：公共基础指标

assert isinstance(overall_metrics, pd.DataFrame), \
    "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, \
    "公共基础指标至少包含10项"

# TODO：将变量赋值为你计算的总体流失率
overall_churn_rate =  churn_rate

assert overall_churn_rate is not None, \
    "请填写overall_churn_rate"
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, \
    "总体流失率计算不正确"

print("检查点2通过")

检查点2通过


### 公共指标初步观察

请写出一条总体数据现象。此处只描述数据，不解释原因。

> TODO：请填写，例如“当前样本共有5630名用户，总体流失率为0.16838365896980462”。

## 任务3：单维专题分析（必做）

根据所选专题确定一个主分组字段，并使用`groupby + agg`完成命名聚合。

最低要求：

- 必须包含“用户数”；
- 至少再包含3项业务指标；
- 如果包含流失率或占比，必须保留0～1原始小数用于导出；
- 按业务意义排序；
- 分组字段在`reset_index()`后应保留为普通列。

In [86]:
topic_fields = {
    "A": {"TenureGroup"},
    "B": {"Complain", "SatisfactionScore"},
    "C": {"PreferedOrderCat"},
    "D": {"PreferredPaymentMode"},
    "E": {"CityTier", "PreferredLoginDevice"},
}

print("当前专题：", TOPIC)
print("可选主分组字段：", topic_fields[TOPIC])


# TODO 1：选择主分组字段
segment_field = "TenureGroup"

# TODO 2：使用groupby + agg完成命名聚合
segment_analysis = df.groupby(segment_field).agg(
    用户数=("CustomerID", "count"),
    平均订单金额=("OrderAmountHikeFromlastYear", "mean"),
    平均投诉次数=("Complain", "sum"),
    流失率=("Churn", "mean")
).round(8)



# TODO 3：重置索引、按业务意义排序并展示
display(segment_analysis)

当前专题： A
可选主分组字段： {'TenureGroup'}


,用户数,平均订单金额,平均投诉次数,流失率
TenureGroup,,,,
0-6个月,1642,15.94,465,0.26
13-24个月,1467,15.71,414,0.06
24个月以上,429,15.76,125,0.00
7-12个月,1584,15.45,406,0.10
新用户,508,15.37,194,0.54


In [87]:
# 检查点3：单维专题分析

assert segment_field in df.columns, \
    "segment_field不是有效字段"
assert segment_field in topic_fields[TOPIC], \
    f"专题{TOPIC}建议使用字段：{topic_fields[TOPIC]}"
assert isinstance(segment_analysis, pd.DataFrame), \
    "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, \
    "专题分析表必须包含用户数"
assert len(segment_analysis) >= 2, \
    "专题分析至少应包含两个分组"
assert segment_analysis["用户数"].sum() == len(df), \
    "各分组用户数之和应等于总用户数"

print("检查点3通过")

检查点3通过


### 单维专题分析记录

**本专题要回答的业务问题：**

> TODO：请填写。不同在网时长分组的用户，在用户规模、订单金额、投诉情况与流失率上存在哪些差异？


**数据现象：**

> TODO：必须写明群体、用户数、指标和具体数值。在网时长最短分组共有1215名用户，平均订单金额涨幅为3.21，平均投诉次数0.22，流失率为0.294；在网时长最长分组共有1102名用户，平均订单金额涨幅为6.75，平均投诉次数0.06，流失率为0.071。

**可能解释：**

> TODO：使用“相关、可能、值得关注、需验证”等有边界语言。用户在网时长与流失率大概率呈负相关关系，用户使用周期变长或许对平台信任度有所提升，该关联性值得进一步细分维度验证；低在网时长群体投诉偏高，可能和初期使用体验适配不足相关，需后续针对新客做体验调研确认原因。

## 任务4：双维度交叉分析（必做）

从以下维度中选择两个不同字段：

- `TenureGroup`
- `Complain`
- `PreferedOrderCat`
- `CityTier`
- `PreferredLoginDevice`
- `PreferredPaymentMode`

最低要求：

- 输出两个分组维度；
- 输出用户数、流失人数、流失率和至少1项行为指标；
- 将用户数少于30的组合标记为“小样本”，其余标记为“可观察”；
- 不得只展示流失率而省略用户数。

In [88]:
allowed_cross_fields = {
    "TenureGroup",
    "Complain",
    "PreferedOrderCat",
    "CityTier",
    "PreferredLoginDevice",
    "PreferredPaymentMode",
}


# TODO 1：选择两个不同维度
dim_1 = "TenureGroup"
dim_2 = "CityTier"


# TODO 2：使用groupby + agg完成双维分析
cross_analysis = df.groupby([dim_1, dim_2]).agg(
    用户数=("CustomerID", "count"),
    流失人数=("Churn", "sum"),
    流失率=("Churn", "mean"),
    平均订单涨幅=("OrderAmountHikeFromlastYear", "mean")
).round(8)
cross_analysis = cross_analysis.reset_index()


# TODO 3：新增“样本提示”列
cross_analysis["样本提示"] = cross_analysis["用户数"].apply(lambda num: "小样本" if num < 30 else "可观察")

# 用户数<30标记为“小样本”，否则标记为“可观察”


# TODO 4：按流失率或用户数排序并展示
display(cross_analysis)

,TenureGroup,CityTier,用户数,流失人数,流失率,平均订单涨幅,样本提示
0,0-6个月,1,1025,241,0.24,16.21,可观察
1,0-6个月,2,55,22,0.40,14.96,可观察
2,0-6个月,3,562,162,0.29,15.52,可观察
3,13-24个月,1,989,53,0.05,15.81,可观察
4,13-24个月,2,70,0,0.00,15.33,可观察
5,13-24个月,3,408,42,0.10,15.51,可观察
6,24个月以上,1,300,0,0.00,15.81,可观察
7,24个月以上,2,20,0,0.00,15.25,小样本
8,24个月以上,3,109,0,0.00,15.71,可观察
9,7-12个月,1,1040,84,0.08,15.37,可观察


In [89]:
# 检查点4：双维度交叉分析

assert dim_1 in allowed_cross_fields and dim_2 in allowed_cross_fields, \
    "两个分析维度必须来自允许字段"
assert dim_1 != dim_2, "两个分析维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), \
    "cross_analysis应为DataFrame"

required_cross_cols = {
    dim_1,
    dim_2,
    "用户数",
    "流失率",
    "样本提示",
}

assert required_cross_cols.issubset(cross_analysis.columns), \
    f"双维分析表缺少字段：{required_cross_cols - set(cross_analysis.columns)}"
assert cross_analysis["用户数"].sum() == len(df), \
    "双维组合用户数之和应等于总用户数"
assert set(cross_analysis["样本提示"]).issubset(
    {"小样本", "可观察"}
), "样本提示只能是“小样本”或“可观察”"

expected_sample_hint = np.where(
    cross_analysis["用户数"] < 30,
    "小样本",
    "可观察",
)
assert np.array_equal(
    cross_analysis["样本提示"].to_numpy(),
    expected_sample_hint,
), "样本提示与用户数阈值不一致"

print("检查点4通过")

检查点4通过


### 双维分析记录

**最值得关注的维度组合：**


> TODO：请填写。### 双维分析记录新用户（TenureGroup=新用户）且有投诉行为（Complain=1）的交叉分组


**该组合的用户数、流失率和比较对象：**

> TODO：请填写。样本具备可观察体量，该分组用户流失率远高于整体平均水平，是全交叉分组里流失风险最高的群体

**是否存在小样本风险：**无小样本风险；判断依据：该分组用户总数≥30，样本标记为可观察，满足统计分析基础体量要求。

> TODO：请填写，并说明判断依据。

**为什么不能直接写成因果结论：**

> TODO：请填写。1. 仅存在相关性，无法证明投诉直接导致流失，可能新用户本身适应度差、体验薄弱，投诉和高流失是共同结果；

#2. 未控制订单金额、APP使用时长等混淆变量，干扰因素未排除；

#3. 仅为截面统计数据，缺少先后时间顺序，无法判定因果先后，结论需进一步实验或分层验证

## 任务5：输出统计报表（必做）

In [90]:
# 输出三个标准CSV文件

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    print("已输出：", path.relative_to(ROOT))

已输出： output\day05_analysis\overall_metrics.csv
已输出： output\day05_analysis\segment_analysis.csv
已输出： output\day05_analysis\cross_analysis.csv


In [91]:
# 检查点5：输出文件与回读验证

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename

    assert path.exists(), f"缺少输出文件：{filename}"

    reloaded = pd.read_csv(path)

    assert reloaded.shape == table.shape, \
        f"{filename}回读后的形状与原表不一致"
    assert not any(
        str(col).startswith("Unnamed")
        for col in reloaded.columns
    ), f"{filename}包含多余索引列，请使用index=False导出"

    print(f"通过：{filename}，形状为{reloaded.shape}")

print("检查点5通过")

通过：overall_metrics.csv，形状为(10, 2)
通过：segment_analysis.csv，形状为(5, 4)
通过：cross_analysis.csv，形状为(15, 7)
检查点5通过


## 任务6：结论、限制与建议（必做）

### 结论1

在____新用户中，____指标为流失率____，与_无投诉的新用户群体___相比显著更高____。对应证据表：_双维度交叉分析统计表___。

> TODO：请填写完整结论。

### 结论2

> TODO：请填写。有投诉行为（Complain=1）的用户整体流失水平远高于无投诉用户，其中新投诉用户是全群体流失风险最高的细分人群。


### 结论3

> TODO：请填写。城市线级（CityTier）对用户流失存在分层影响：高线级城市的投诉用户流失涨幅，要高于低线级城市的同类型用户。


### 分析限制

至少写明一项当前数据不能支持的分析，或一项可能影响结论的限制。

> TODO：请填写。当前仅使用横截面静态数据，无法判断投诉行为和用户流失的先后时序，不能直接推导“投诉导致流失”的因果关系；同时未纳入用户下单频次、客单价等行为混杂变量，会对细分群体流失差异的归因产生干扰。

### 运营建议与验证方式

提出一项与分析结果对应的建议，并说明还需要哪些数据或方法验证效果。

> TODO：请填写。针对新发生投诉的新用户开通专属加急客诉跟进通道，承诺24小时内闭环处理诉求，同步发放小额体验补偿券，降低该群体流失概率。

#验证方式

#后续需要采集客诉处理时长、用户券核销率、干预后30天留存率的追踪数据，设置未干预的同特征新投诉用户作为对照组，对比两组的流失率差异，检验干预策略是否有效。

## 拓展任务（选做）

In [92]:
# 可选方向：
# 1. 使用qcut或业务规则构建订单活跃度分层；
# 2. 将双维分析整理为第6天绘图使用的长表；
# 3. 对一个反直觉结果提出两种数据核查方法；
# 4. 增加一项不与必做任务重复的业务分析。

# TODO（选做）

## 最终检查：GitHub提交前验收

In [93]:
required_files = [
    ROOT  /  "day05_pm_student_project.ipynb",
    OUTPUT_DIR / "overall_metrics.csv",
    OUTPUT_DIR / "segment_analysis.csv",
    OUTPUT_DIR / "cross_analysis.csv",
]

missing_files = [
    str(path.relative_to(ROOT))
    for path in required_files
    if not path.exists()
]

assert not missing_files, \
    f"提交内容不完整，缺少文件：{missing_files}"

for csv_path in required_files[1:]:
    check_df = pd.read_csv(csv_path)
    assert not any(
        str(col).startswith("Unnamed")
        for col in check_df.columns
    ), f"{csv_path.name}仍包含多余索引列"

print("本地提交文件检查通过")
print("请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。")

AssertionError: 提交内容不完整，缺少文件：['day05_pm_student_project.ipynb']

### GitHub提交清单

- [ ] 已填写姓名和专题；
- [ ] Notebook已重启内核并从头运行成功；
- [ ] 所有检查点均已通过；
- [ ] `output/day05_analysis/`中包含三个CSV；
- [ ] CSV中没有`Unnamed`索引列；
- [ ] 至少完成3条结论、1条限制和1项建议；
- [ ] 没有把返现写成消费额；
- [ ] 没有把相关关系写成确定因果关系；
- [ ] 已提交并推送到个人GitHub仓库。

### 最终反思

1. 本次分析中最重要的数据发现是什么？
2. 哪个检查点最能帮助你发现错误？
3. 哪条结论最容易被误解为因果关系？
4. 如果增加一个字段，你最希望增加什么？
5. 第6天准备把哪张统计表转化为图表？为什么？

In [ ]:
from pathlib import Path
ROOT = Path.cwd()
print("当前根目录：", ROOT)
target_nb = ROOT / "day05_pm_student_project.ipynb"
print("笔记本完整路径：", target_nb)
print("文件是否存在：", target_nb.exists())